In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------------------------
# 1. LOAD DATASET
# -------------------------------------------------------------------------
# Load raw crop yield data
df = pd.read_excel("crop_yield_cleaned.xlsx")


# -------------------------------------------------------------------------
# 2. FEATURE ENGINEERING (Domain-Specific Transformations)
# -------------------------------------------------------------------------
def create_crop_features(data):
    df_feat = data.copy()

    # --- A. Soil Nutrient Dynamics (NPK) ---
    # Total primary macro-nutrient availability
    df_feat["total_npk"] = (
        df_feat["nitrogen_n"] + df_feat["phosphorus_p"] + df_feat["potassium_k"]
    )

    # Relative nutrient proportions (%)
    df_feat["n_proportion"] = df_feat["nitrogen_n"] / df_feat["total_npk"]
    df_feat["p_proportion"] = df_feat["phosphorus_p"] / df_feat["total_npk"]
    df_feat["k_proportion"] = df_feat["potassium_k"] / df_feat["total_npk"]

    # Key nutrient ratios (critical for balanced crop nutrition)
    df_feat["np_ratio"] = df_feat["nitrogen_n"] / (
        df_feat["phosphorus_p"] + 1e-5
    )
    df_feat["nk_ratio"] = df_feat["nitrogen_n"] / (
        df_feat["potassium_k"] + 1e-5
    )

    # --- B. Climate & Agronomic Interactions ---
    # Rain-to-Temperature Ratio (Moisture availability vs. Evapotranspiration demand)
    df_feat["rain_temp_ratio"] = df_feat["rain_fall_mm"] / (
        df_feat["temperatue"] + 1e-5
    )

    # Quadratic temperature term (Captures non-linear heat stress response curves)
    df_feat["temp_squared"] = df_feat["temperatue"] ** 2

    # Synergy between water availability and commercial fertilizer application
    df_feat["rain_fertilizer_interaction"] = (
        df_feat["rain_fall_mm"] * df_feat["fertilizer"]
    )

    return df_feat


# Apply engineered features
df_engineered = create_crop_features(df)


# -------------------------------------------------------------------------
# 3. CATEGORICAL ENCODING
# -------------------------------------------------------------------------
# One-Hot Encoding for categorical crop type variables
crop_dummies = pd.get_dummies(
    df_engineered["crop_type"], prefix="crop", drop_first=True
)
df_engineered = pd.concat(
    [df_engineered.drop(columns=["crop_type"]), crop_dummies], axis=1
)


# -------------------------------------------------------------------------
# 4. TRAIN / TEST SPLIT & FEATURE SCALING
# -------------------------------------------------------------------------
# Separate target variable (yeild_qacre) and features
X = df_engineered.drop(columns=["yeild_qacre"])
y = df_engineered["yeild_qacre"]

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features for distance-sensitive or linear algorithms
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns
)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Engineered Training Feature Matrix Shape:", X_train_scaled.shape)
print("\nFirst 5 Rows of Engineered Dataset:")
print(X_train_scaled.head())

Engineered Training Feature Matrix Shape: (799, 20)

First 5 Rows of Engineered Dataset:
   rain_fall_mm  fertilizer  temperatue  nitrogen_n  phosphorus_p  \
0      0.376032   -1.617481   -0.622150   -1.315985     -0.915941   
1     -1.564105   -0.611748    0.585235   -0.024250      1.684254   
2     -0.579313    0.034795   -0.967117   -0.024250     -0.915941   
3      0.536347   -1.258291    0.757719    1.502345     -0.443178   
4      1.174335   -0.899100    1.275170    0.093180     -0.679559   

   potassium_k  total_npk  n_proportion  p_proportion  k_proportion  np_ratio  \
0    -0.464658  -1.591120     -0.016769     -0.188095      0.211194 -0.027410   
1    -1.461335   0.098946     -0.183902      1.837964     -1.611381 -1.207720   
2     1.030356   0.005053     -0.049194     -1.042550      1.113677  0.733234   
3     1.279526   1.507333      0.409286     -1.171387      0.647103  1.030577   
4     1.030356   0.192838     -0.112353     -0.863063      1.015104  0.529425   

   nk_rat

---

## Breakdown of Key Engineered Features

| Feature Category | Engineered Column | Mathematical Formula | Agronomic Significance |
| --- | --- | --- | --- |
| **Nutrient Balance** | `total_npk` | $N + P + K$ | Measures total essential macronutrient concentration. |
| **Nutrient Ratios** | `np_ratio`, `nk_ratio` | $\frac{N}{P}$, $\frac{N}{K}$ | Nutrient stoichiometry; crop response often depends on balance rather than raw amounts. |
| **Moisture Efficiency** | `rain_temp_ratio` | $\frac{\text{Rainfall}}{\text{Temperature}}$ | Proxies effective moisture available to plants after accounting for temperature-driven evaporation. |
| **Input Synergy** | `rain_fertilizer_interaction` | $\text{Rainfall} \times \text{Fertilizer}$ | Captures how soil water availability boosts or restricts chemical fertilizer uptake. |
| **Thermal Response** | `temp_squared` | $\text{Temperature}^2$ | Models non-linear yield decline at high heat thresholds. |